<a href="https://colab.research.google.com/github/atikur234/automation-learning/blob/main/day2chromaDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.6 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling openteleme

In [6]:
import chromadb

# 1. Initialize the persistent client (saves data to a folder)
client = chromadb.PersistentClient(path="./chroma_db")

# 2. Create a collection
collection = client.get_or_create_collection(name="my_knowledge_base")

# 3. Add your Day 1 documents
# Note: Chroma handles the embedding part automatically!
# collection.add(
#     documents=[
#         "To make a great beef curry, you need slow-cooked onions and authentic spices.",
#         "Python is a versatile programming language used heavily in AI and Data Science.",
#         "The traffic in Dhaka can be challenging during peak office hours."
#     ],
#     ids=["id1", "id2", "id3"] # Every doc needs a unique ID
# )

# 4. Query the DB
results = collection.query(
    query_texts=[" favourite curry?"],
    n_results=1
)

print(results)

{'ids': [['id1']], 'embeddings': None, 'documents': [['To make a great beef curry, you need slow-cooked onions and authentic spices.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None]], 'distances': [[1.0709636211395264]]}


In [10]:
# Adding data with Metadata
documents = [
    "The ZARQE Summer Collection features 100% organic cotton shirts in pastel colors.",
    "Our Winter '26 Jackets are water-resistant and rated for temperatures down to 5°C.",
    "Standard delivery within Dhaka takes 24-48 hours via our local courier partners.",
    "International shipping to the USA and Europe takes 7-12 business days.",
    "Return Policy: Items can be returned within 14 days if the tags are intact.",
    "Customer Review: The blue linen shirt is very breathable, perfect for Dhaka heat.",
    "Customer Review: Delivery to Chittagong was slow, took almost 5 days.",
    "The ZARQE flagship store is located in Banani, Dhaka, open daily from 10 AM.",
    "Wholesale inquiries should be directed to sales@zarqe.com for bulk pricing.",
    "Our fabrics are sourced from sustainable mills in Gazipur and Narayanganj."
]

metadatas = [
    {"category": "product", "season": "summer", "location": "global"},
    {"category": "product", "season": "winter", "location": "global"},
    {"category": "logistics", "type": "delivery", "location": "Dhaka"},
    {"category": "logistics", "type": "delivery", "location": "international"},
    {"category": "policy", "type": "returns", "location": "global"},
    {"category": "review", "sentiment": "positive", "location": "Dhaka"},
    {"category": "review", "sentiment": "negative", "location": "Chittagong"},
    {"category": "info", "type": "store", "location": "Dhaka"},
    {"category": "info", "type": "business", "location": "global"},
    {"category": "info", "type": "production", "location": "Bangladesh"}
]

ids = [f"zarqe_{i}" for i in range(len(documents))]

# Add to your collection
collection.add(documents=documents, metadatas=metadatas, ids=ids)

# Querying with a Metadata Filter
# This tells the DB: "Ignore everything except the shipping policy"
results = collection.query(
    query_texts=["How long for delivery?"],
    where={"$and": [{"category": "logistics"}, {"type": "delivery"}]},
    n_results=1
)
print(results)

{'ids': [['zarqe_2']], 'embeddings': None, 'documents': [['Standard delivery within Dhaka takes 24-48 hours via our local courier partners.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'location': 'Dhaka', 'category': 'logistics', 'type': 'delivery'}]], 'distances': [[0.8837448954582214]]}


In [11]:
results = collection.query(
    query_texts=["How long does it take?"],
    where={"category": "policy"}, # Only looks at the return policy doc
    n_results=1
)
print(results)

{'ids': [['zarqe_4']], 'embeddings': None, 'documents': [['Return Policy: Items can be returned within 14 days if the tags are intact.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'type': 'returns', 'location': 'global', 'category': 'policy'}]], 'distances': [[1.327192783355713]]}


In [13]:
results = collection.query(
    query_texts=["What should I wear?"],
    where={"location": "Dhaka"}, # Should find the summer collection or store info
    n_results=2
)
print(results)

{'ids': [['zarqe_5', 'zarqe_7']], 'embeddings': None, 'documents': [['Customer Review: The blue linen shirt is very breathable, perfect for Dhaka heat.', 'The ZARQE flagship store is located in Banani, Dhaka, open daily from 10 AM.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'location': 'Dhaka', 'sentiment': 'positive', 'category': 'review'}, {'location': 'Dhaka', 'category': 'info', 'type': 'store'}]], 'distances': [[1.4082845449447632, 2.021329402923584]]}


In [16]:
results = collection.query(
    query_texts=["shirt"],
    where={"$and": [{"category": "logistics"}, {"location": "international"}]},
    n_results=1

)
print(results)

{'ids': [['zarqe_3']], 'embeddings': None, 'documents': [['International shipping to the USA and Europe takes 7-12 business days.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'location': 'international', 'category': 'logistics', 'type': 'delivery'}]], 'distances': [[2.022125005722046]]}
